# 4e. ARR bridge and revenue bridge

**What an investor reads into this.** The ARR bridge splits the change in run-rate between one December and the next into what was won (new customers, cross-sell, upsell, price) and what was lost (downgrades, contraction, churn). It shows whether growth comes from selling more to more customers or from putting prices up on a shrinking base. The revenue bridge does the same for reported revenue year on year, which answers investor question 6 directly: what explains the change in revenue from last year to this year.

Definitions: `docs/metric_definitions.md` (ARR bridge and revenue bridge sections). ARR = December MRR × 12. The price-increase factor k for each year is the uniform same-line change shown in 4a.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nb_utils import connect, q, sql_file, save_table
import charts as ch

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
con = connect()
dl = sql_file(con, "dec_mrr_customer_line")
dl["arr"] = 12 * dl.dec_mrr
k = sql_file(con, "uplift_factors").set_index("year").k
years = [int(y) for y in k.index]

## 1. Evidence for the price-increase component

Same-line % change in MRR between consecutive Decembers, for every customer x product line held in both (from `sql/analysis/price_change_distribution.sql`). Every continuing line moves by exactly the same percentage in each year, so the whole same-line change is price; tier upsells and same-line contraction are zero. Invoice amounts are rounded to the penny, so a line can differ from opening × k by a penny; residuals below £1 of ARR per line are counted in the price increase.

In [2]:
pcd = sql_file(con, "price_change_distribution")
pcd

,year,pct_change,lines,share_of_lines_pct
0,2023,0.0,1425,100.0
1,2024,5.0,1686,100.0
2,2025,7.0,1890,100.0


## 2. ARR bridge

In [3]:
ROUNDING = 1.00  # £ of ARR per line
comp_order = ["opening_arr", "new", "cross_sell", "upsell", "price_increase", "downgrade", "contraction", "churn", "closing_arr"]
bridge, detail = [], []
for y in years:
    o = dl[dl.year == y - 1].set_index(["customer_id", "product_line"]).arr
    c = dl[dl.year == y].set_index(["customer_id", "product_line"]).arr
    oc, cc = set(o.index.get_level_values(0)), set(c.index.get_level_values(0))
    new = c[c.index.get_level_values(0).isin(cc - oc)].sum()
    churn = -o[o.index.get_level_values(0).isin(oc - cc)].sum()
    cont = oc & cc
    oo = o[o.index.get_level_values(0).isin(cont)]; ccn = c[c.index.get_level_values(0).isin(cont)]
    both = oo.index.intersection(ccn.index)
    cross = ccn[ccn.index.difference(oo.index)].sum()
    down = -oo[oo.index.difference(ccn.index)].sum()
    # Invoice amounts are rounded to the penny, so a line's closing MRR can differ from
    # opening x k by a penny (12p of ARR). Residuals below ROUNDING are part of the price increase.
    resid = ccn[both] - oo[both] * float(k[y])
    real = resid.abs() >= ROUNDING
    price = (oo[both] * (float(k[y]) - 1)).sum() + resid[~real].sum()
    upsell, contraction = resid[real & (resid > 0)].sum(), resid[real & (resid < 0)].sum()
    row = {"year": y, "opening_arr": o.sum(), "new": new, "cross_sell": cross, "upsell": upsell, "price_increase": price,
           "downgrade": down, "contraction": contraction, "churn": churn, "closing_arr": c.sum(),
           "new_customers": len(cc - oc), "churned_customers": len(oc - cc), "continuing_customers": len(cont),
           "cross_sell_lines": len(ccn.index.difference(oo.index)), "downgrade_lines": len(oo.index.difference(ccn.index)),
           "lines_with_upsell": int((real & (resid > 0)).sum()), "lines_with_contraction": int((real & (resid < 0)).sum()),
           "uplift_factor": float(k[y])}
    row["check_difference"] = row["opening_arr"] + sum(row[x] for x in comp_order[1:-1]) - row["closing_arr"]
    assert abs(row["check_difference"]) < 0.01, row
    bridge.append(row)
arr_bridge = save_table(pd.DataFrame(bridge), "4e_arr_bridge")
arr_bridge[comp_order + ["check_difference"]].set_index(arr_bridge.year).T.round(0)

year,2023,2024,2025
opening_arr,29005684.0,32781343.0,37496836.0
new,5003640.0,5126228.0,5571544.0
cross_sell,360985.0,471698.0,413062.0
upsell,0.0,0.0,0.0
price_increase,0.0,1518997.0,2314269.0
downgrade,-79064.0,-314614.0,-1529874.0
contraction,0.0,0.0,0.0
churn,-1509903.0,-2086815.0,-2906009.0
closing_arr,32781343.0,37496836.0,41359828.0
check_difference,0.0,0.0,-0.0


In [4]:
# Long form for the databook and dashboard
long = arr_bridge.melt(id_vars="year", value_vars=comp_order, var_name="component", value_name="arr")
long["order"] = long.component.map({c: i for i, c in enumerate(comp_order)})
save_table(long.sort_values(["year", "order"]), "4e_arr_bridge_long")
arr_bridge[["year", "new_customers", "churned_customers", "continuing_customers", "cross_sell_lines", "downgrade_lines",
            "lines_with_upsell", "lines_with_contraction"]]

,year,new_customers,churned_customers,continuing_customers,cross_sell_lines,downgrade_lines,lines_with_upsell,lines_with_contraction
0,2023,285,90,920,53,12,0,0
1,2024,270,132,1073,61,16,0,0
2,2025,290,171,1172,50,17,0,0


## 3. Waterfall charts

In [5]:
labels = {"opening_arr": "Opening\nARR", "new": "New", "cross_sell": "Cross-\nsell", "upsell": "Upsell", "price_increase": "Price\nincrease",
          "downgrade": "Down-\ngrade", "contraction": "Contrac-\ntion", "churn": "Churn", "closing_arr": "Closing\nARR"}
def delta_label(v):
    if abs(v) < 0.5:
        return "£0"
    sign = "+" if v >= 0 else "-"
    return f"{sign}£{abs(v) / 1e6:.2f}m" if abs(v) >= 100_000 else f"{sign}£{abs(v) / 1e3:.0f}k"

lo_all = min(min(r["opening_arr"], r["closing_arr"]) for r in bridge) * 0.85
hi_all = max(max(r["opening_arr"], r["closing_arr"]) + max(r["new"], 0) for r in bridge) * 1.06
fig, axes = plt.subplots(1, len(years), figsize=(15, 4.8), sharey=True)
for ax, r in zip(axes, bridge):
    running = 0.0
    for i, comp in enumerate(comp_order):
        v = r[comp]
        if comp in ("opening_arr", "closing_arr"):
            ax.bar(i, v, color=ch.TOTAL, width=0.7)
            running = v if comp == "opening_arr" else running
            ax.text(i, v * 1.005, f"£{v / 1e6:.1f}m", ha="center", va="bottom", fontsize=7.5, color=ch.TEXT)
        else:
            bottom = running if v >= 0 else running + v
            ax.bar(i, abs(v), bottom=bottom, color=ch.INCREASE if v >= 0 else ch.DECREASE, width=0.7)
            ax.text(i, (bottom + abs(v) if abs(v) >= 0.5 else running) + 150_000, delta_label(v), ha="center", va="bottom",
                    fontsize=7, color=ch.TEXT if abs(v) >= 0.5 else ch.TEXT_2)
            running += v
    ax.set_xticks(range(len(comp_order))); ax.set_xticklabels([labels[c] for c in comp_order], fontsize=7)
    ax.set_ylim(lo_all, hi_all)
    ax.yaxis.set_major_formatter(ch.gbp_m(0))
    ax.set_title(f"{r['year']}: December {r['year'] - 1} to December {r['year']}", fontsize=10)
axes[0].set_ylabel("ARR (£m; axis does not start at zero)")
fig.suptitle("ARR bridge by year (blue: increase, red: decrease, grey: opening and closing ARR)", x=0.01, ha="left", fontsize=11, fontweight="bold")
fig.tight_layout(rect=(0, 0, 1, 0.94))
print(ch.save(fig, "4e_arr_bridge"))

outputs/charts/4e_arr_bridge.png


## 4. Revenue bridge (year on year, for investor question 6)

Recurring billings are analysed on annual totals at customer x product line. For lines billed in both years: price effect = billings in Y × (1 − 1/k); upsell / contraction = months billed in Y × (price per month in Y / k − price per month in Y−1); months effect = (months in Y − months in Y−1) × price per month in Y−1. A positive months effect is the full-year effect of customers or lines added part-way through Y−1. A negative one is a part-year loss: churn if the customer has no MRR in December Y, downgrade if the line ends but the customer stays, and a billing gap if the line is still live in December.

In [6]:
rl = sql_file(con, "recurring_line_year")
tot = q(con, """SELECT year(month_start) AS year, sum(net_revenue) AS net_revenue, sum(credit_notes) AS credit_notes,
                       sum(oneoff_signup_net) AS signup_implementation, sum(oneoff_other_net) AS other_implementation,
                       sum(recurring_net) AS recurring_net
                FROM fact_revenue_monthly GROUP BY 1""").set_index("year")
rows = []
for y in years:
    p = rl[rl.year == y - 1].set_index(["customer_id", "product_line"])
    c = rl[rl.year == y].set_index(["customer_id", "product_line"])
    pc = p.groupby(level=0).billings.sum(); cc = c.groupby(level=0).billings.sum()
    pcust, ccust = set(pc[pc > 0].index), set(cc[cc > 0].index)
    comp = dict.fromkeys(["new_customers", "full_year_effect_of_prior_year_additions", "cross_sell", "upsell", "price_increase",
                          "billing_gaps", "downgrade", "contraction", "churn"], 0.0)
    comp["new_customers"] = cc[list(ccust - pcust)].sum()
    comp["churn"] -= pc[list(pcust - ccust)].sum()
    cont = pcust & ccust
    pp = p[p.index.get_level_values(0).isin(cont)]; cp = c[c.index.get_level_values(0).isin(cont)]
    comp["cross_sell"] += cp.loc[cp.index.difference(pp.index)].billings.sum()
    comp["downgrade"] -= pp.loc[pp.index.difference(cp.index)].billings.sum()
    both = pp.index.intersection(cp.index)
    kk = float(k[y])
    for idx in both:
        a, b = pp.loc[idx], cp.loc[idx]
        if a.billings == 0 and b.billings == 0:
            continue
        unit_prev = a.billings / a.months_billed if a.months_billed else 0.0
        unit_curr = b.billings / b.months_billed if b.months_billed else 0.0
        price = b.billings * (1 - 1 / kk)
        up = b.months_billed * (unit_curr / kk - unit_prev)
        if abs(up) < ROUNDING:  # penny rounding of invoice amounts, as in the ARR bridge
            price += up
            up = 0.0
        months = (b.months_billed - a.months_billed) * unit_prev
        comp["price_increase"] += price
        if up >= 0: comp["upsell"] += up
        else: comp["contraction"] += up
        if months > 0:
            comp["full_year_effect_of_prior_year_additions"] += months
        elif months < 0:
            if not b.customer_live_in_december: comp["churn"] += months
            elif not b.line_live_in_december: comp["downgrade"] += months
            else: comp["billing_gaps"] += months
    row = {"year": y, "net_revenue_prior": tot.loc[y - 1, "net_revenue"], **comp,
           "credit_notes_change": tot.loc[y, "credit_notes"] - tot.loc[y - 1, "credit_notes"],
           "signup_implementation_change": tot.loc[y, "signup_implementation"] - tot.loc[y - 1, "signup_implementation"],
           "other_implementation_change": tot.loc[y, "other_implementation"] - tot.loc[y - 1, "other_implementation"],
           "net_revenue": tot.loc[y, "net_revenue"]}
    rec_parts = list(comp) + ["credit_notes_change"]
    row["recurring_change"] = sum(row[x] for x in rec_parts)
    row["oneoff_change"] = row["signup_implementation_change"] + row["other_implementation_change"]
    row["check_difference"] = row["net_revenue_prior"] + row["recurring_change"] + row["oneoff_change"] - row["net_revenue"]
    row["check_recurring"] = row["recurring_change"] - (tot.loc[y, "recurring_net"] - tot.loc[y - 1, "recurring_net"])
    assert abs(row["check_difference"]) < 0.01 and abs(row["check_recurring"]) < 0.01, row
    rows.append(row)
rev_bridge = save_table(pd.DataFrame(rows), "4e_revenue_bridge")
rev_bridge.set_index("year").T.round(0)

year,2023,2024,2025
net_revenue_prior,28232152.0,32192870.0,37258899.0
new_customers,2884584.0,2982907.0,3068356.0
full_year_effect_of_prior_year_additions,2039768.0,2273217.0,2183164.0
cross_sell,196903.0,278900.0,278671.0
upsell,0.0,404.0,0.0
price_increase,-0.0,1579728.0,2455688.0
billing_gaps,-3912.0,-1753.0,-1741.0
downgrade,-76395.0,-196871.0,-896690.0
contraction,0.0,0.0,0.0
churn,-1326723.0,-1772195.0,-2639572.0


In [7]:
rb_order = ["net_revenue_prior", "new_customers", "full_year_effect_of_prior_year_additions", "cross_sell", "upsell", "price_increase",
            "billing_gaps", "downgrade", "contraction", "churn", "credit_notes_change", "signup_implementation_change",
            "other_implementation_change", "net_revenue"]
rbl = rev_bridge.melt(id_vars="year", value_vars=rb_order, var_name="component", value_name="amount")
rbl["group"] = rbl.component.map(lambda c: "total" if c in ("net_revenue_prior", "net_revenue")
                                 else "one-off" if "implementation" in c else "recurring")
rbl["order"] = rbl.component.map({c: i for i, c in enumerate(rb_order)})
save_table(rbl.sort_values(["year", "order"]), "4e_revenue_bridge_long")

,year,component,amount,group,order
0,2023,net_revenue_prior,2.823215e+07,total,0
3,2023,new_customers,2.884584e+06,recurring,1
6,2023,full_year_effect_of_prior_year_additions,2.039768e+06,recurring,2
9,2023,cross_sell,1.969026e+05,recurring,3
12,2023,upsell,0.000000e+00,recurring,4
15,2023,price_increase,-1.392664e-11,recurring,5
18,2023,billing_gaps,-3.911950e+03,recurring,6
21,2023,downgrade,-7.639550e+04,recurring,7
24,2023,contraction,0.000000e+00,recurring,8
27,2023,churn,-1.326723e+06,recurring,9


In [8]:
latest = rev_bridge.iloc[-1]
short = {"net_revenue_prior": f"{int(latest.year) - 1}\nrevenue", "new_customers": "New\ncustomers",
         "full_year_effect_of_prior_year_additions": "Full-year\neffect", "cross_sell": "Cross-\nsell", "upsell": "Upsell",
         "price_increase": "Price\nincrease", "billing_gaps": "Billing\ngaps", "downgrade": "Down-\ngrade", "contraction": "Contrac-\ntion",
         "churn": "Churn", "credit_notes_change": "Credit\nnotes", "signup_implementation_change": "Signup\nimpl.",
         "other_implementation_change": "Other\nimpl.", "net_revenue": f"{int(latest.year)}\nrevenue"}
fig, ax = plt.subplots(figsize=(12, 4.8))
running = 0.0
for i, comp in enumerate(rb_order):
    v = float(latest[comp])
    if comp in ("net_revenue_prior", "net_revenue"):
        ax.bar(i, v, color=ch.TOTAL, width=0.7); running = v if i == 0 else running
        ax.text(i, v * 1.003, f"£{v / 1e6:.1f}m", ha="center", va="bottom", fontsize=7.5)
    else:
        bottom = running if v >= 0 else running + v
        colour = (ch.AQUA if "implementation" in comp else ch.INCREASE) if v >= 0 else ch.DECREASE
        ax.bar(i, abs(v), bottom=bottom, color=colour, width=0.7)
        ax.text(i, (bottom + abs(v) if abs(v) >= 0.5 else running) + 120_000, delta_label(v), ha="center", va="bottom", fontsize=7,
                color=ch.TEXT if abs(v) >= 0.5 else ch.TEXT_2)
        running += v
ax.set_xticks(range(len(rb_order))); ax.set_xticklabels([short[c] for c in rb_order], fontsize=7.5)
ax.set_ylim(latest.net_revenue_prior * 0.85, latest.net_revenue * 1.06)
ax.yaxis.set_major_formatter(ch.gbp_m(0))
ax.set_ylabel("Net revenue (£m; axis does not start at zero)")
ax.set_title(f"Revenue bridge, {int(latest.year) - 1} to {int(latest.year)} (blue: recurring increase, aqua: one-off increase, red: decrease)")
print(ch.save(fig, "4e_revenue_bridge"))

outputs/charts/4e_revenue_bridge.png


## 5. Notes on the components

In [9]:
# Every "new" customer in the ARR bridge is new to Northbridge: none had been billed before the opening December.
returning = q(con, """
    WITH d AS (SELECT DISTINCT customer_id, year(month_start) AS y FROM fact_mrr_monthly WHERE month(month_start) = 12)
    SELECT b.y AS year, count(*) AS new_in_bridge,
           count(*) FILTER (WHERE EXISTS (SELECT 1 FROM fact_mrr_monthly m WHERE m.customer_id = b.customer_id
                                          AND m.month_start < make_date(b.y - 1, 12, 1))) AS billed_before_opening_december
    FROM d b WHERE b.y > 2022 AND NOT EXISTS (SELECT 1 FROM d a WHERE a.customer_id = b.customer_id AND a.y = b.y - 1)
    GROUP BY 1 ORDER BY 1
""")
save_table(returning, "4e_new_customers_check")
returning

,year,new_in_bridge,billed_before_opening_december
0,2023,285,0
1,2024,270,0
2,2025,290,0


In [10]:
# Lines behind any non-zero upsell or contraction in the revenue bridge: a line that lapsed and was re-subscribed
# at a different contracted price (the ARR bridge counts these as cross-sell, because the line was not held in the opening December).
resub = q(con, """
    SELECT s.customer_id, p.product_line, s.subscription_id, s.start_date, s.end_date, s.monthly_price
    FROM clean_subscriptions s JOIN clean_products p USING (product_id)
    WHERE (s.customer_id, s.product_id) IN (SELECT customer_id, product_id FROM clean_subscriptions GROUP BY ALL HAVING count(*) > 1)
    ORDER BY 1, 4
""")
save_table(resub, "4e_resubscribed_lines")
print(f"customer x product pairs with more than one contract line: {resub.groupby(['customer_id', 'product_line']).ngroups}")
resub.head(10)

customer x product pairs with more than one contract line: 14


,customer_id,product_line,subscription_id,start_date,end_date,monthly_price
0,C1021,Analytics Add-on,S50039,2017-02-22,2018-06-01,465.50
1,C1021,Analytics Add-on,S50040,2019-12-01,2025-03-01,465.50
2,C1083,Analytics Add-on,S50155,2020-11-01,2021-12-01,517.03
3,C1083,Analytics Add-on,S50156,2022-10-01,NaT,517.03
4,C1088,Analytics Add-on,S50166,2017-06-07,2019-02-01,521.66
5,C1088,Analytics Add-on,S50167,2022-04-01,NaT,521.66
6,C1103,Analytics Add-on,S50192,2017-08-25,2022-04-01,520.50
7,C1103,Analytics Add-on,S50193,2024-01-01,NaT,546.52
8,C1104,Analytics Add-on,S50195,2017-08-14,2019-06-01,495.58
9,C1104,Analytics Add-on,S50196,2025-08-01,NaT,530.27
